# Notebook 5 — Exercises

Practise everything from Notebooks 1–4 on the same café data.

**How to use this notebook**
1. Run the **Setup** cell first.
2. For each exercise, read the task, then write your code in the empty cell below it.
3. Stuck? Open the **Hint**. Finished? Open the **Solution** to check your work.
4. Write your interpretation in the **Your answer** cell — explaining the result matters as much as the number.

| Part | Topic | Notebook |
|---|---|---|
| A | Exploring relationships | 1 |
| B | Covariance and correlation | 2 |
| C | Correlation vs causation | 3 |
| D | Regression and machine learning | 4 |

## Setup
Loads the libraries and all three datasets: `cafe` (daily data), `exp` (promo experiment) and `stores` (two locations).

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")

cafe   = pd.read_csv("cafe_daily.csv", parse_dates=["date"])
exp    = pd.read_csv("cafe_promo_experiment.csv")
stores = pd.read_csv("cafe_two_stores.csv")
cafe.head()

## Part A — Exploring Relationships (Notebook 1)

### Exercise A1 — Independent and dependent variables

The café manager wants to know whether **more customers lead to more pastries sold**.

1. Which column is the **independent** variable ($x$) and which is the **dependent** variable ($y$)?
2. Make a scatter plot with $x$ on the x-axis and $y$ on the y-axis. Label both axes.
3. Describe the plot using **direction**, **strength** and **form**.

In [ ]:
# Exercise A1 — your code here
x_col = ...   # independent variable (column name as a string)
y_col = ...   # dependent variable

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

The two columns are `foot_traffic` (customers per day) and `pastry_demand`. Use `sns.scatterplot(data=cafe, x=..., y=...)`.

</details>

<details>
<summary><b>Solution</b></summary>

```python
x_col = 'foot_traffic'     # independent: number of customers
y_col = 'pastry_demand'    # dependent: pastries sold
sns.scatterplot(data=cafe, x=x_col, y=y_col, alpha=.4)
plt.xlabel('Foot traffic (customers/day) — independent')
plt.ylabel('Pastry demand — dependent')
plt.title('Customers vs pastries'); plt.show()
```

- $x$ = `foot_traffic`, $y$ = `pastry_demand` (pastries depend on how many people come in).
- **Direction:** positive — more customers, more pastries.
- **Strength:** fairly strong — the points form a clear band, with some spread.
- **Form:** linear (a straight band, no curve). No obvious outliers.

</details>

### Exercise A2 — Reading four scatter plots

Make a 1 × 3 grid of scatter plots:
- `temperature_c` vs `cold_brew_sales`
- `temperature_c` vs `salad_sales`
- `hot_drink_sales` vs `iced_drink_sales`

For each one, write down the **direction** and **strength**, and guess whether $r$ will be close to +1, −1 or 0.

In [ ]:
# Exercise A2 — your code here
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
# sns.scatterplot(data=cafe, x=..., y=..., ax=ax[0])

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

Use `ax=ax[0]`, `ax=ax[1]`, `ax=ax[2]` to place each plot in its own panel.

</details>

<details>
<summary><b>Solution</b></summary>

```python
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.scatterplot(data=cafe, x='temperature_c', y='cold_brew_sales', alpha=.4, ax=ax[0])
sns.scatterplot(data=cafe, x='temperature_c', y='salad_sales', alpha=.4, ax=ax[1])
sns.scatterplot(data=cafe, x='hot_drink_sales', y='iced_drink_sales', alpha=.4, ax=ax[2])
ax[0].set_title('temp vs cold brew'); ax[1].set_title('temp vs salad'); ax[2].set_title('hot vs iced drinks')
plt.tight_layout(); plt.show()
```

- Temperature vs cold brew: strong positive ($r \approx +0.95$).
- Temperature vs salad: strong positive ($r \approx +0.92$).
- Hot vs iced drinks: strong **negative** ($r \approx -0.81$) — on warm days people switch from hot to iced drinks.
  (Temperature is behind this one too!)

</details>

## Part B — Covariance and Correlation (Notebook 2)

### Exercise B1 — Covariance by hand

Calculate the covariance between `temperature_c` and `cold_brew_sales` **using the formula**, not `np.cov`:

$$\operatorname{cov}(x, y) = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})$$

**Where:** $x_i, y_i$ are the values on day $i$, $\bar{x}, \bar{y}$ are the means, and $n$ is the number of days.

Then check your answer against `np.cov(x, y)[0, 1]`.

In [ ]:
# Exercise B1 — your code here
x = cafe['temperature_c']
y = cafe['cold_brew_sales']
n = len(x)
cov_manual = ...
print(cov_manual)

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

`x - x.mean()` gives all the deviations at once. Multiply the two deviation series, `.sum()` them, and divide by `n - 1`.

</details>

<details>
<summary><b>Solution</b></summary>

```python
x = cafe['temperature_c']
y = cafe['cold_brew_sales']
n = len(x)
cov_manual = ((x - x.mean()) * (y - y.mean())).sum() / (n - 1)
print(f'manual  : {cov_manual:.2f}')
print(f'np.cov  : {np.cov(x, y)[0, 1]:.2f}')
```

Both give about **142.86**. The value is positive, so temperature and cold-brew sales move in the same direction.

</details>

### Exercise B2 — Pearson's r by hand

Use your covariance from B1 to calculate Pearson's $r$:

$$r = \frac{\operatorname{cov}(x, y)}{s_x\, s_y}$$

**Where:** $s_x$ and $s_y$ are the standard deviations of $x$ and $y$ (pandas: `.std()`, which uses $n-1$).

1. Compare your answer with `stats.pearsonr(x, y)`.
2. Calculate $r^2$. What share of the variation in cold-brew sales does temperature explain?

In [ ]:
# Exercise B2 — your code here
r_manual = ...
print(r_manual)

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

`x.std()` and `y.std()` give $s_x$ and $s_y$. $r^2$ is just `r_manual ** 2`.

</details>

<details>
<summary><b>Solution</b></summary>

```python
r_manual = cov_manual / (x.std() * y.std())
r_scipy, _ = stats.pearsonr(x, y)
print(f'manual r : {r_manual:+.3f}')
print(f'scipy r  : {r_scipy:+.3f}')
print(f'r squared: {r_manual**2:.3f}')
```

$r \approx +0.948$ and $r^2 \approx 0.90$: temperature explains about **90 %** of the day-to-day variation in cold-brew sales.

</details>

### Exercise B3 — Do units matter?

Convert temperature from °C to °F using $F = \tfrac{9}{5}C + 32$.

**Where:** $C$ is the temperature in degrees Celsius and $F$ the same temperature in degrees Fahrenheit.

Recalculate the **covariance** and the **correlation** between temperature and `cold_brew_sales` using °F.
Which one changed, and why?

In [ ]:
# Exercise B3 — your code here
temp_f = ...

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

Use `np.cov(temp_f, y)[0, 1]` and `temp_f.corr(y)`.

</details>

<details>
<summary><b>Solution</b></summary>

```python
temp_f = cafe['temperature_c'] * 9 / 5 + 32
print(f'cov (°C) = {np.cov(x, y)[0, 1]:.2f}   r (°C) = {x.corr(y):+.3f}')
print(f'cov (°F) = {np.cov(temp_f, y)[0, 1]:.2f}   r (°F) = {temp_f.corr(y):+.3f}')
```

The covariance grows from about 143 to about 257 (multiplied by 9/5), but $r$ stays at **+0.948**.
Covariance depends on the units; correlation divides by the standard deviations, so the units cancel out.

</details>

### Exercise B4 — Outlier check

Row 40 is the one-off catering day. Calculate $r$ between `temperature_c` and `daily_revenue`
**with** and **without** that row. Does the outlier make the relationship look stronger or weaker?

In [ ]:
# Exercise B4 — your code here

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

`cafe.drop(40)` returns the table without row 40.

</details>

<details>
<summary><b>Solution</b></summary>

```python
r_with    = cafe['temperature_c'].corr(cafe['daily_revenue'])
no_out    = cafe.drop(40)
r_without = no_out['temperature_c'].corr(no_out['daily_revenue'])
print(f'with outlier   : {r_with:+.3f}')
print(f'without outlier: {r_without:+.3f}')
```

$r$ rises from about **0.63** to about **0.66** without the outlier. That single cold day with very high revenue
was making the relationship look *weaker* than it really is.

</details>

### Exercise B5 — Splitting a curve

In Notebook 2, discount vs profit had a weak overall $r \approx -0.34$, even though the plot showed a clear arc.

Calculate $r$ between `discount_pct` and `daily_profit` separately for:
- days with a discount of **15 % or less**
- days with a discount of **20 % or more**

What do the two results tell you about the shape?

In [ ]:
# Exercise B5 — your code here
low  = cafe[cafe['discount_pct'] <= 15]
high = ...

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

Filter with `cafe[cafe['discount_pct'] >= 20]`, then use `.corr()` on each part.

</details>

<details>
<summary><b>Solution</b></summary>

```python
low  = cafe[cafe['discount_pct'] <= 15]
high = cafe[cafe['discount_pct'] >= 20]
print(f'discount <= 15%: r = {low.discount_pct.corr(low.daily_profit):+.3f}')
print(f'discount >= 20%: r = {high.discount_pct.corr(high.daily_profit):+.3f}')
```

About **+0.90** for small discounts and **−0.94** for large discounts. Each half is a strong straight line, but in
opposite directions — when combined they cancel out, which is why the overall $r$ looked weak.
The best discount is somewhere around 15–20 %.

</details>

## Part C — Correlation and Causation (Notebook 3)

### Exercise C1 — Partial correlation from the formula

In Notebook 3 we controlled for temperature using residuals. Now use the **formula** instead:

$$r_{xy\cdot z} = \frac{r_{xy} - r_{xz}\,r_{yz}}{\sqrt{(1-r_{xz}^2)(1-r_{yz}^2)}}$$

**Where:** $x$ = `cold_brew_sales`, $y$ = `salad_sales`, $z$ = `temperature_c`; $r_{xy}$, $r_{xz}$, $r_{yz}$ are the
ordinary correlations between each pair, and $r_{xy\cdot z}$ is the partial correlation of $x$ and $y$ controlling for $z$.

Do you get the same answer as Notebook 3 (about −0.016)?

In [ ]:
# Exercise C1 — your code here
r_xy = cafe['cold_brew_sales'].corr(cafe['salad_sales'])
r_xz = ...
r_yz = ...

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

Compute the three correlations with `.corr()`, then plug them into the formula with `np.sqrt`.

</details>

<details>
<summary><b>Solution</b></summary>

```python
r_xy = cafe['cold_brew_sales'].corr(cafe['salad_sales'])
r_xz = cafe['cold_brew_sales'].corr(cafe['temperature_c'])
r_yz = cafe['salad_sales'].corr(cafe['temperature_c'])
partial = (r_xy - r_xz * r_yz) / np.sqrt((1 - r_xz**2) * (1 - r_yz**2))
print(f'r_xy = {r_xy:+.3f}, r_xz = {r_xz:+.3f}, r_yz = {r_yz:+.3f}')
print(f'partial r = {partial:+.3f}')
```

Yes — the partial correlation is about **−0.016**, the same as the residual method. Both methods remove temperature's
influence; once it is gone, cold brew and salad are unrelated.

</details>

### Exercise C2 — Is it a shared trend?

`foot_traffic` and `social_followers` are correlated. Is the café's social media bringing people in,
or are both just growing over time?

1. Calculate the raw $r$.
2. Detrend both series with $x_t^{\text{detrended}} = x_t - \hat{\beta}_1\, t$ and calculate $r$ again.

**Where:** $t$ is the day number (0 … 730), $x_t$ the value on day $t$, and $\hat{\beta}_1$ the slope of the line fitted
between the series and $t$.

In [ ]:
# Exercise C2 — your code here
t = np.arange(len(cafe))

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

`stats.linregress(t, series).slope` gives $\hat{\beta}_1$. Subtract `slope * t` from the series.

</details>

<details>
<summary><b>Solution</b></summary>

```python
t = np.arange(len(cafe))
def detrend(series):
    return series - stats.linregress(t, series).slope * t
r_raw = cafe['foot_traffic'].corr(cafe['social_followers'])
r_det = np.corrcoef(detrend(cafe['foot_traffic']), detrend(cafe['social_followers']))[0, 1]
print(f'raw r       = {r_raw:+.3f}')
print(f'detrended r = {r_det:+.3f}')
```

The raw $r \approx +0.63$ drops to about **−0.09** after detrending. The correlation came from both series growing
over time — there is no evidence that followers bring in customers.

</details>

### Exercise C3 — Fixing the observational promo estimate

In the observational data, promos were mostly run on weekends, so the naive gap (~€74) was overstated.
One way to control for a confounder is to **compare like with like** (stratify):

1. Calculate $\bar{y}_{\text{promo}} - \bar{y}_{\text{no promo}}$ separately for **weekdays** and for **weekends**.
2. Combine the two using a weighted average, weighting each group by its number of days:
   $$\text{effect} = \frac{n_{\text{weekday}}\,\text{effect}_{\text{weekday}} + n_{\text{weekend}}\,\text{effect}_{\text{weekend}}}{n_{\text{weekday}} + n_{\text{weekend}}}$$

**Where:** $\bar{y}$ is average daily revenue, $n_{\text{weekday}}$ and $n_{\text{weekend}}$ are the number of days in each group,
and $\text{effect}_{\text{group}}$ is the promo gap inside that group.

How does your estimate compare with the naive €74 and with the experimental result (~€48, true effect €40)?

In [ ]:
# Exercise C3 — your code here
for weekend, group in cafe.groupby('is_weekend'):
    ...

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

Inside the loop: `group[group.promo_active == 1].daily_revenue.mean()` minus the same for `== 0`. Store each gap and `len(group)`.

</details>

<details>
<summary><b>Solution</b></summary>

```python
effects, sizes = [], []
for weekend, group in cafe.groupby('is_weekend'):
    gap = (group[group.promo_active == 1].daily_revenue.mean()
           - group[group.promo_active == 0].daily_revenue.mean())
    effects.append(gap); sizes.append(len(group))
    print(f"{'weekend' if weekend else 'weekday'}: promo gap = €{gap:.0f}  ({len(group)} days)")
combined = np.average(effects, weights=sizes)
print(f'stratified estimate = €{combined:.0f}')
```

About €24 on weekdays and €59 on weekends, giving a stratified estimate of about **€34**. That is much closer to the
true €40 than the naive €74, because we no longer compare promo weekends against normal weekdays.
The two groups still differ a lot because the data is noisy — which is why a randomised experiment is more reliable.

</details>

### Exercise C4 — Explaining Simpson's paradox

Using `stores`, calculate the **average** `staff_on_shift` and `customer_satisfaction` for each store.
Use the numbers to explain, in your own words, why the pooled correlation is negative while each store's correlation is positive.

In [ ]:
# Exercise C4 — your code here

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

`stores.groupby('store')[['staff_on_shift', 'customer_satisfaction']].mean()`

</details>

<details>
<summary><b>Solution</b></summary>

```python
stores.groupby('store')[['staff_on_shift', 'customer_satisfaction']].mean().round(1)
```

Downtown averages about 9.8 staff but only ~66.6 satisfaction; the Suburb averages ~4.0 staff with ~84.8 satisfaction.
When pooled, the high-staff points all belong to the low-satisfaction store, so the overall trend slopes **down**.
Inside each store, adding staff still raises satisfaction. The **store** is the confounder.

</details>

## Part D — Regression and Machine Learning (Notebook 4)

### Exercise D1 — Regression line from correlation

Predict `pastry_demand` ($y$) from `foot_traffic` ($x$) **without** scikit-learn, using

$$\beta_1 = r\,\frac{s_y}{s_x}, \qquad \beta_0 = \bar{y} - \beta_1\,\bar{x}, \qquad \hat{y} = \beta_0 + \beta_1 x$$

**Where:** $r$ is the correlation of $x$ and $y$, $s_x, s_y$ their standard deviations, $\bar{x}, \bar{y}$ their means,
$\beta_0$ the intercept, $\beta_1$ the slope and $\hat{y}$ the predicted value.

1. Calculate $\beta_1$ and $\beta_0$ and compare them with `LinearRegression`.
2. Predict pastry demand for a day with **300 customers**.
3. Calculate $R^2 = r^2$. Is foot traffic a better predictor of pastries than temperature was ($R^2 \approx 0.10$)?

In [ ]:
# Exercise D1 — your code here
x = cafe['foot_traffic']
y = cafe['pastry_demand']
b1 = ...
b0 = ...

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

`r = x.corr(y)`, then `b1 = r * y.std() / x.std()` and `b0 = y.mean() - b1 * x.mean()`.

</details>

<details>
<summary><b>Solution</b></summary>

```python
from sklearn.linear_model import LinearRegression
x = cafe['foot_traffic']
y = cafe['pastry_demand']
r  = x.corr(y)
b1 = r * y.std() / x.std()
b0 = y.mean() - b1 * x.mean()
lin = LinearRegression().fit(cafe[['foot_traffic']], y)
print(f'manual : y_hat = {b0:.2f} + {b1:.3f} x')
print(f'sklearn: y_hat = {lin.intercept_:.2f} + {lin.coef_[0]:.3f} x')
print(f'prediction at 300 customers: {b0 + b1 * 300:.0f} pastries')
print(f'R² = r² = {r**2:.3f}')
```

$\hat{y} \approx 10.0 + 0.251x$ (both methods agree). At 300 customers: about **85 pastries**.
$R^2 \approx 0.57$ — foot traffic explains about 57 % of the variation, far more than temperature's 10 %.

</details>

### Exercise D2 — Does a useless feature help?

Using the features from Notebook 4, train a `LinearRegression` model to predict `daily_revenue` twice:
once **with** and once **without** `barista_shoe_size`. Use the same `train_test_split(..., test_size=.25, random_state=0)`
and compare the test $R^2$:

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

**Where:** $y_i$ is the actual revenue on test day $i$, $\hat{y}_i$ the predicted revenue, and $\bar{y}$ the mean revenue.

In [ ]:
# Exercise D2 — your code here
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
feats = ['temperature_c','iced_drink_sales','hot_drink_sales','cold_brew_sales',
         'salad_sales','foot_traffic','barista_shoe_size','pastry_demand']

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Hint</b></summary>

Write a small function that takes a list of features, splits, fits and returns `model.score(X_test, y_test)`.

</details>

<details>
<summary><b>Solution</b></summary>

```python
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
feats = ['temperature_c','iced_drink_sales','hot_drink_sales','cold_brew_sales',
         'salad_sales','foot_traffic','barista_shoe_size','pastry_demand']
def test_r2(features):
    Xtr, Xte, ytr, yte = train_test_split(cafe[features], cafe['daily_revenue'],
                                          test_size=.25, random_state=0)
    return LinearRegression().fit(Xtr, ytr).score(Xte, yte)
print(f'with shoe size   : R² = {test_r2(feats):.4f}')
print(f'without shoe size: R² = {test_r2([f for f in feats if f != "barista_shoe_size"]):.4f}')
```

The two scores are practically identical. `barista_shoe_size` has $r \approx 0$ with revenue, so it adds nothing —
removing it makes the model simpler without losing accuracy.

</details>

### Exercise D3 — Reflection

Answer in a few sentences (no code needed):

1. A model predicts revenue very well using `iced_drink_sales`. Does that mean selling more iced drinks will *cause* revenue to rise? What else could be going on?
2. Name one situation in your own life or work where two things are correlated but one probably does not cause the other.

**Your answer:** _double-click to write your interpretation here._

<details>
<summary><b>Solution</b></summary>

1. Not necessarily. Iced drinks are part of revenue, but both are also driven by temperature (a confounder) and by
   foot traffic. Pushing iced drinks on a cold day would not bring the same gain. Only an experiment can show the causal effect.
2. Answers will vary — e.g. ice-cream sales and sunburn (both caused by sunny weather).

</details>

## Well done!
You have now measured relationships, tested whether they are real, and used them to make predictions.
Remember the three big lessons: **plot first**, **correlation is not causation**, and **check for confounders**.